In [5]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
GPU: Tesla T4


In [6]:
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 200)
        self.fc2 = nn.Linear(200, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = MNISTNet().to(device)

print("Model device:", next(model.parameters()).device)

Model device: cuda:0


In [7]:

import torch

# Create 64 sample MNIST images
images = torch.rand(64, 1, 28, 28).to(device)

# Test the model
model.eval()

with torch.no_grad():
    output = model(images)

print("Input shape:", images.shape)
print("Output shape:", output.shape)
print("Model device:", next(model.parameters()).device)
print("Test successful!")


Input shape: torch.Size([64, 1, 28, 28])
Output shape: torch.Size([64, 10])
Model device: cuda:0
Test successful!


In [8]:

import torch
import torch.nn.functional as F

def linear_bounds(lower, upper, layer):
    center = (lower + upper) / 2
    radius = (upper - lower) / 2

    new_center = F.linear(
        center, layer.weight, layer.bias
    )
    new_radius = F.linear(
        radius, layer.weight.abs()
    )

    return new_center - new_radius, new_center + new_radius


def ibp_loss(model, images, labels, perturbation, tau=0.01):

    center = images + perturbation

    lower = torch.clamp(center - tau, 0, 1)
    upper = torch.clamp(center + tau, 0, 1)

    lower = lower.view(lower.size(0), -1)
    upper = upper.view(upper.size(0), -1)

    lower, upper = linear_bounds(lower, upper, model.fc1)

    lower = F.relu(lower)
    upper = F.relu(upper)

    lower, upper = linear_bounds(lower, upper, model.fc2)

    true_lower = lower.gather(1, labels[:, None])
    worst_margins = upper - true_lower

    worst_margins = worst_margins.scatter(
        1, labels[:, None], 0.0
    )

    return torch.logsumexp(worst_margins, dim=1)


# Test IBP using 64 sample images
images = torch.rand(64, 1, 28, 28).to(device)
labels = torch.randint(0, 10, (64,)).to(device)

perturbation = torch.zeros(1, 1, 28, 28).to(device)

losses = ibp_loss(
    model, images, labels, perturbation, tau=0.01
)

print("IBP loss shape:", losses.shape)
print("Average IBP loss:", losses.mean().item())
print("IBP test successful!")


IBP loss shape: torch.Size([64])
Average IBP loss: 3.140066146850586
IBP test successful!


In [9]:

import time

def generate_perturbation(model, images, labels,
                          epsilon=0.1, tau=0.01, steps=3):
    budget = epsilon - tau
    delta = torch.zeros_like(images)

    for _ in range(steps):
        delta = delta.detach().requires_grad_(True)

        logits = model(torch.clamp(images + delta, 0, 1))
        loss = F.cross_entropy(logits, labels)

        grad = torch.autograd.grad(loss, delta)[0]

        delta = (
            delta + (budget / steps * 2) * grad.sign()
        ).clamp(-budget, budget).detach()

    return delta


def balanced_citrus_loss(model, images, labels,
                         epsilon=0.1, tau=0.01,
                         alpha=0.5, num_sources=2):

    batch_size = images.size(0)

    source_indices = torch.randperm(
        batch_size, device=images.device
    )[:num_sources]

    all_losses = []

    for j in source_indices:
        j = int(j.item())

        v_j = generate_perturbation(
            model,
            images[j:j+1],
            labels[j:j+1],
            epsilon,
            tau
        ).detach()

        losses = ibp_loss(
            model, images, labels, v_j, tau
        )

        # Exclude the source image itself
        mask = torch.ones(
            batch_size, dtype=torch.bool,
            device=images.device
        )
        mask[j] = False

        all_losses.append((losses, mask))

    loss_matrix = torch.stack([x[0] for x in all_losses])
    valid_mask = torch.stack([x[1] for x in all_losses])

    avg_loss = (
        (loss_matrix * valid_mask).sum(dim=0)
        / valid_mask.sum(dim=0).clamp(min=1)
    )

    hard_loss = loss_matrix.masked_fill(
        ~valid_mask, float("-inf")
    ).max(dim=0).values

    valid_images = valid_mask.any(dim=0)

    proposed_loss = (
        (1 - alpha) * avg_loss[valid_images]
        + alpha * hard_loss[valid_images]
    )

    return proposed_loss.mean()


# Test execution time
model.train()

torch.cuda.synchronize()
start = time.time()

loss = balanced_citrus_loss(
    model, images, labels,
    epsilon=0.1,
    tau=0.01,
    alpha=0.5,
    num_sources=2
)

loss.backward()

torch.cuda.synchronize()
elapsed = time.time() - start

print(f"Balanced CITRUS Loss: {loss.item():.4f}")
print(f"Execution Time: {elapsed:.2f} seconds")
print("Test successful!")


Balanced CITRUS Loss: 3.1169
Execution Time: 0.40 seconds
Test successful!


In [11]:

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Download MNIST training dataset
train_data = datasets.MNIST(
    root="./mnist_data",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

# Download MNIST test dataset
test_data = datasets.MNIST(
    root="./mnist_data",
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

# Create data loaders
train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=False
)

print("Training images:", len(train_data))
print("Test images:", len(test_data))
print("Training batches:", len(train_loader))
print("Data loading successful!")


100%|██████████| 9.91M/9.91M [00:00<00:00, 20.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 500kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.70MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.3MB/s]

Training images: 60000
Test images: 10000
Training batches: 938
Data loading successful!


In [12]:

import time
import torch

# Create optimizer for the current model
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 2

for epoch in range(epochs):
    model.train()
    total_loss = 0
    start = time.time()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        loss = balanced_citrus_loss(
            model,
            images,
            labels,
            epsilon=0.1,
            tau=0.01,
            alpha=0.5,
            num_sources=2
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Average Loss: {total_loss/len(train_loader):.4f} | "
        f"Time: {(time.time()-start)/60:.2f} minutes"
    )

print("Training completed!")


Epoch 1/2 | Average Loss: 0.8810 | Time: 0.30 minutes
Epoch 2/2 | Average Loss: 0.5083 | Time: 0.37 minutes
Training completed!


In [13]:

def evaluate_clean(model, test_loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total


clean_accuracy = evaluate_clean(model, test_loader)

print(f"Standard Accuracy: {clean_accuracy:.2f}%")


Standard Accuracy: 92.15%


In [14]:

import torch
import torch.nn.functional as F

def evaluate_uap(model, test_loader, epsilon=0.1,
                 steps=100, step_size=0.01):

    model.eval()

    # One universal perturbation for all images
    delta = torch.zeros(
        1, 1, 28, 28, device=device
    )

    # Generate a universal perturbation
    for step in range(steps):
        images, labels = next(iter(test_loader))
        images = images.to(device)
        labels = labels.to(device)

        delta = delta.detach().requires_grad_(True)

        outputs = model(
            torch.clamp(images + delta, 0, 1)
        )

        loss = F.cross_entropy(outputs, labels)

        gradient = torch.autograd.grad(
            loss, delta
        )[0]

        delta = (
            delta + step_size * gradient.sign()
        ).clamp(-epsilon, epsilon).detach()

    # Evaluate the same perturbation on all test images
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(
                torch.clamp(images + delta, 0, 1)
            )

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total


uap_accuracy = evaluate_uap(
    model,
    test_loader,
    epsilon=0.1,
    steps=100,
    step_size=0.01
)

print(f"Standard Accuracy: {clean_accuracy:.2f}%")
print(f"UAP Attack Accuracy: {uap_accuracy:.2f}%")


Standard Accuracy: 92.15%
UAP Attack Accuracy: 87.22%


In [15]:

torch.save(
    model.state_dict(),
    "balanced_citrus_alpha05.pth"
)

print("Model saved successfully!")


Model saved successfully!


In [16]:

# Create a new model for the baseline
baseline_model = MNISTNet().to(device)

baseline_optimizer = torch.optim.Adam(
    baseline_model.parameters(),
    lr=0.001
)

epochs = 2

for epoch in range(epochs):
    baseline_model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        baseline_optimizer.zero_grad()

        loss = balanced_citrus_loss(
            baseline_model,
            images,
            labels,
            epsilon=0.1,
            tau=0.01,
            alpha=0.0,
            num_sources=2
        )

        loss.backward()
        baseline_optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Average Loss: {total_loss/len(train_loader):.4f}"
    )

print("Baseline training completed!")


Epoch 1/2 | Average Loss: 0.8373
Epoch 2/2 | Average Loss: 0.4649
Baseline training completed!


In [17]:

# Evaluate baseline standard accuracy
baseline_clean = evaluate_clean(
    baseline_model,
    test_loader
)

# Evaluate baseline UAP attack accuracy
baseline_uap = evaluate_uap(
    baseline_model,
    test_loader,
    epsilon=0.1,
    steps=100,
    step_size=0.01
)

print("=== Model Comparison ===")
print(f"Balanced CITRUS Standard: {clean_accuracy:.2f}%")
print(f"Baseline Standard:        {baseline_clean:.2f}%")
print()
print(f"Balanced CITRUS UAP:      {uap_accuracy:.2f}%")
print(f"Baseline UAP:             {baseline_uap:.2f}%")


=== Model Comparison ===
Balanced CITRUS Standard: 92.15%
Baseline Standard:        92.21%

Balanced CITRUS UAP:      87.22%
Baseline UAP:             89.79%


In [18]:

import torch
import time

# Set random seed
torch.manual_seed(42)

# Create a new model
model_alpha01 = MNISTNet().to(device)

optimizer_alpha01 = torch.optim.Adam(
    model_alpha01.parameters(),
    lr=0.001
)

epochs = 2

for epoch in range(epochs):
    model_alpha01.train()
    total_loss = 0
    start = time.time()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer_alpha01.zero_grad()

        loss = balanced_citrus_loss(
            model_alpha01,
            images,
            labels,
            epsilon=0.1,
            tau=0.01,
            alpha=0.1,
            num_sources=2
        )

        loss.backward()
        optimizer_alpha01.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Average Loss: {total_loss/len(train_loader):.4f} | "
        f"Time: {(time.time()-start)/60:.2f} minutes"
    )

print("Alpha 0.1 training completed!")


Epoch 1/2 | Average Loss: 0.8026 | Time: 0.29 minutes
Epoch 2/2 | Average Loss: 0.4519 | Time: 0.31 minutes
Alpha 0.1 training completed!


In [19]:

# Standard accuracy
alpha01_clean = evaluate_clean(
    model_alpha01,
    test_loader
)

# UAP attack accuracy
alpha01_uap = evaluate_uap(
    model_alpha01,
    test_loader,
    epsilon=0.1,
    steps=100,
    step_size=0.01
)

print("=== Model Comparison ===")
print(f"Alpha=0.0 Standard: {baseline_clean:.2f}%")
print(f"Alpha=0.1 Standard: {alpha01_clean:.2f}%")
print(f"Alpha=0.5 Standard: {clean_accuracy:.2f}%")

print()

print(f"Alpha=0.0 UAP: {baseline_uap:.2f}%")
print(f"Alpha=0.1 UAP: {alpha01_uap:.2f}%")
print(f"Alpha=0.5 UAP: {uap_accuracy:.2f}%")


=== Model Comparison ===
Alpha=0.0 Standard: 92.21%
Alpha=0.1 Standard: 92.85%
Alpha=0.5 Standard: 92.15%

Alpha=0.0 UAP: 89.79%
Alpha=0.1 UAP: 87.90%
Alpha=0.5 UAP: 87.22%


In [20]:

torch.save(
    model_alpha01.state_dict(),
    "balanced_citrus_alpha01.pth"
)

print("Alpha 0.1 model saved!")


Alpha 0.1 model saved!


In [21]:

import torch
import copy
import random
import numpy as np

# Fix random seed
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Create one initial model
initial_model = MNISTNet().to(device)

# Save its initial weights in memory
initial_weights = copy.deepcopy(
    initial_model.state_dict()
)

print("Initial weights saved successfully!")


Initial weights saved successfully!


In [22]:

import torch
import time
import random
import numpy as np

results = {}
trained_models = {}

alpha_values = [0.0, 0.1, 0.5]
epochs = 10

for alpha in alpha_values:

    # Reset random seeds for a fair comparison
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)

    # Create model with identical initial weights
    current_model = MNISTNet().to(device)
    current_model.load_state_dict(initial_weights)

    optimizer = torch.optim.Adam(
        current_model.parameters(),
        lr=0.001
    )

    print(f"\n=== Training alpha={alpha} ===")

    start = time.time()

    for epoch in range(epochs):
        current_model.train()
        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            loss = balanced_citrus_loss(
                current_model,
                images,
                labels,
                epsilon=0.1,
                tau=0.01,
                alpha=alpha,
                num_sources=4
            )

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(
            f"Alpha={alpha} | "
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss={total_loss/len(train_loader):.4f}"
        )

    training_time = (time.time() - start) / 60

    # Save model weights
    torch.save(
        current_model.state_dict(),
        f"balanced_citrus_alpha_{alpha}.pth"
    )

    trained_models[alpha] = current_model

    results[alpha] = {
        "training_time": training_time
    }

    print(
        f"Alpha={alpha} completed in "
        f"{training_time:.2f} minutes"
    )

print("\nAll training completed!")



=== Training alpha=0.0 ===
Alpha=0.0 | Epoch 1/10 | Loss=0.8154
Alpha=0.0 | Epoch 2/10 | Loss=0.4541
Alpha=0.0 | Epoch 3/10 | Loss=0.3865
Alpha=0.0 | Epoch 4/10 | Loss=0.3466
Alpha=0.0 | Epoch 5/10 | Loss=0.3219
Alpha=0.0 | Epoch 6/10 | Loss=0.3046
Alpha=0.0 | Epoch 7/10 | Loss=0.2921
Alpha=0.0 | Epoch 8/10 | Loss=0.2810
Alpha=0.0 | Epoch 9/10 | Loss=0.2725
Alpha=0.0 | Epoch 10/10 | Loss=0.2668
Alpha=0.0 completed in 4.49 minutes

=== Training alpha=0.1 ===
Alpha=0.1 | Epoch 1/10 | Loss=0.8279
Alpha=0.1 | Epoch 2/10 | Loss=0.4688
Alpha=0.1 | Epoch 3/10 | Loss=0.3989
Alpha=0.1 | Epoch 4/10 | Loss=0.3583
Alpha=0.1 | Epoch 5/10 | Loss=0.3337
Alpha=0.1 | Epoch 6/10 | Loss=0.3167
Alpha=0.1 | Epoch 7/10 | Loss=0.3041
Alpha=0.1 | Epoch 8/10 | Loss=0.2931
Alpha=0.1 | Epoch 9/10 | Loss=0.2843
Alpha=0.1 | Epoch 10/10 | Loss=0.2788
Alpha=0.1 completed in 4.51 minutes

=== Training alpha=0.5 ===
Alpha=0.5 | Epoch 1/10 | Loss=0.8686
Alpha=0.5 | Epoch 2/10 | Loss=0.5126
Alpha=0.5 | Epoch 3/10 | Los

In [23]:

print("=== Standard Accuracy After 10 Epochs ===")

for alpha in [0.0, 0.1, 0.5]:
    current_model = trained_models[alpha]

    accuracy = evaluate_clean(
        current_model,
        test_loader
    )

    results[alpha]["standard_accuracy"] = accuracy

    print(
        f"Alpha={alpha}: "
        f"Standard Accuracy = {accuracy:.2f}%"
    )


=== Standard Accuracy After 10 Epochs ===
Alpha=0.0: Standard Accuracy = 95.32%
Alpha=0.1: Standard Accuracy = 95.19%
Alpha=0.5: Standard Accuracy = 95.15%


In [24]:

import torch
import torch.nn.functional as F

def stronger_uap_accuracy(model, train_loader, test_loader,
                          epsilon=0.1, steps=100,
                          step_size=0.01, restarts=3):

    model.eval()
    accuracies = []

    for restart in range(restarts):

        # One shared perturbation
        delta = torch.empty(
            1, 1, 28, 28, device=device
        ).uniform_(-epsilon, epsilon)

        train_iter = iter(train_loader)

        for step in range(steps):
            try:
                images, labels = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                images, labels = next(train_iter)

            images = images.to(device)
            labels = labels.to(device)

            delta = delta.detach().requires_grad_(True)

            outputs = model(
                torch.clamp(images + delta, 0, 1)
            )

            loss = F.cross_entropy(outputs, labels)

            grad = torch.autograd.grad(
                loss, delta
            )[0]

            delta = (
                delta + step_size * grad.sign()
            ).clamp(-epsilon, epsilon).detach()

        # Evaluate on held-out test images
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(
                    torch.clamp(images + delta, 0, 1)
                )

                correct += (
                    outputs.argmax(dim=1) == labels
                ).sum().item()

                total += labels.size(0)

        accuracy = 100 * correct / total
        accuracies.append(accuracy)

        print(
            f"Restart {restart+1}: "
            f"UAP Accuracy = {accuracy:.2f}%"
        )

    return min(accuracies)


print("=== Stronger UAP Evaluation ===")

for alpha in [0.0, 0.1, 0.5]:
    print(f"\nEvaluating alpha={alpha}")

    accuracy = stronger_uap_accuracy(
        trained_models[alpha],
        train_loader,
        test_loader,
        epsilon=0.1,
        steps=100,
        step_size=0.01,
        restarts=3
    )

    results[alpha]["uap_accuracy"] = accuracy

    print(
        f"Alpha={alpha}: "
        f"Lowest UAP Accuracy = {accuracy:.2f}%"
    )


=== Stronger UAP Evaluation ===

Evaluating alpha=0.0
Restart 1: UAP Accuracy = 89.76%
Restart 2: UAP Accuracy = 88.31%
Restart 3: UAP Accuracy = 88.15%
Alpha=0.0: Lowest UAP Accuracy = 88.15%

Evaluating alpha=0.1
Restart 1: UAP Accuracy = 89.07%
Restart 2: UAP Accuracy = 89.45%
Restart 3: UAP Accuracy = 90.01%
Alpha=0.1: Lowest UAP Accuracy = 89.07%

Evaluating alpha=0.5
Restart 1: UAP Accuracy = 89.67%
Restart 2: UAP Accuracy = 88.94%
Restart 3: UAP Accuracy = 90.39%
Alpha=0.5: Lowest UAP Accuracy = 88.94%


In [25]:

import torch
import torch.nn.functional as F

def evaluate_certified(model, test_loader, epsilon=0.1):
    model.eval()

    certified_correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            # All possible L-infinity perturbations
            lower = torch.clamp(images - epsilon, 0, 1)
            upper = torch.clamp(images + epsilon, 0, 1)

            lower = lower.view(lower.size(0), -1)
            upper = upper.view(upper.size(0), -1)

            # First linear layer
            lower, upper = linear_bounds(
                lower, upper, model.fc1
            )

            # ReLU
            lower = F.relu(lower)
            upper = F.relu(upper)

            # Output layer
            lower, upper = linear_bounds(
                lower, upper, model.fc2
            )

            # Lower bound of the correct class
            true_lower = lower.gather(
                1, labels[:, None]
            )

            # Upper bounds of competing classes
            margins = true_lower - upper

            # Ignore the correct class itself
            margins.scatter_(
                1, labels[:, None], float("inf")
            )

            # Certified if all competing margins > 0
            certified = (margins > 0).all(dim=1)

            certified_correct += certified.sum().item()
            total += labels.size(0)

    return 100 * certified_correct / total


print("=== IBP Certified Accuracy ===")

for alpha in [0.0, 0.1, 0.5]:
    accuracy = evaluate_certified(
        trained_models[alpha],
        test_loader,
        epsilon=0.1
    )

    results[alpha]["certified_accuracy"] = accuracy

    print(
        f"Alpha={alpha}: "
        f"Certified Accuracy = {accuracy:.2f}%"
    )


=== IBP Certified Accuracy ===
Alpha=0.0: Certified Accuracy = 6.63%
Alpha=0.1: Certified Accuracy = 6.80%
Alpha=0.5: Certified Accuracy = 5.15%
